In [1]:
# ============================================
# Cell 1: Imports, warnings, and project paths
# Purpose:
# - Load all dependencies required for optimized target construction
# - Keep project paths consistent with the corrected folder structure
# - Prepare a clean environment for target engineering
# ============================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
META_DIR = PROJECT_ROOT / "reports" / "metadata"

for folder in [DATA_DIR, PROCESSED_DIR, META_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_START = "2015-01-01"
TRAIN_END = "2024-12-31"
LIVE_START = "2025-01-01"
LIVE_END = "2026-03-01"

print("Project root:", PROJECT_ROOT)
print("Processed directory:", PROCESSED_DIR)

Project root: c:\Users\Tyler\Regime-Aware Risk Integrated Alpha
Processed directory: c:\Users\Tyler\Regime-Aware Risk Integrated Alpha\data\processed


In [2]:
# ============================================
# Cell 2: Load Phase 2.5 selected optimized feature dataset
# Purpose:
# - Load the optimized feature package created in Phase 2.5
# - Preserve time ordering
# - Confirm the new optimized input contract before target construction
# ============================================

phase_2_5_parquet = PROCESSED_DIR / "phase_2_5_selected_feature_data.parquet"
phase_2_5_csv = PROCESSED_DIR / "phase_2_5_selected_feature_data.csv"

print("Looking for Phase 2.5 selected feature files:")
print("Parquet:", phase_2_5_parquet)
print("CSV:", phase_2_5_csv)

if phase_2_5_parquet.exists():
    df = pd.read_parquet(phase_2_5_parquet)
    print("Loaded Parquet file")
elif phase_2_5_csv.exists():
    df = pd.read_csv(phase_2_5_csv, index_col=0, parse_dates=True)
    print("Loaded CSV file")
else:
    raise FileNotFoundError(
        "Phase 2.5 selected feature data was not found. "
        f"Expected one of:\n{phase_2_5_parquet}\n{phase_2_5_csv}"
    )

df.index = pd.to_datetime(df.index)
df = df.sort_index()

print("\nDataset shape:", df.shape)
print("\nDataset preview:")
print(df.head())

Looking for Phase 2.5 selected feature files:
Parquet: c:\Users\Tyler\Regime-Aware Risk Integrated Alpha\data\processed\phase_2_5_selected_feature_data.parquet
CSV: c:\Users\Tyler\Regime-Aware Risk Integrated Alpha\data\processed\phase_2_5_selected_feature_data.csv
Loaded Parquet file

Dataset shape: (2780, 53)

Dataset preview:
              spy_open    spy_high     spy_low   spy_close  spy_adj_close  \
Date                                                                        
2015-02-02  200.050003  202.029999  197.860001  201.919998     167.218262   
2015-02-03  203.000000  204.850006  202.550003  204.839996     169.636398   
2015-02-04  203.919998  205.380005  203.509995  204.059998     168.990479   
2015-02-05  204.860001  206.300003  204.770004  206.119995     170.696411   
2015-02-06  206.559998  207.240005  204.919998  205.550003     170.224442   

            spy_volume  vix_close  tnx_close  is_train_period  is_live_period  \
Date                                            

In [3]:
# ============================================
# Cell 3: Validation before optimized target construction
# Purpose:
# - Confirm the selected optimized dataset contains the required market and feature columns
# - Fail early if the feature optimization handoff is broken
# - Keep the target layer explicit and reproducible
# ============================================

required_cols = [
    "spy_open",
    "spy_high",
    "spy_low",
    "spy_close",
    "spy_adj_close",
    "spy_volume",
    "vix_close",
    "tnx_close",
    "phase_2_5_regime_proxy",
]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for Phase 3.1: {missing_cols}")

assert df.index.is_monotonic_increasing, "Index must be sorted ascending."
assert df.index.duplicated().sum() == 0, "Duplicate timestamps are not allowed."

print("Phase 3.1 validation passed successfully.")

Phase 3.1 validation passed successfully.


In [4]:
# ============================================
# Cell 4: Helper functions for optimized target construction
# Purpose:
# - Centralize target logic
# - Keep target thresholds transparent and easy to tune later
# - Reduce label noise compared with the original Phase 3
# ============================================

def classify_three_way_signal(forward_return: float, buy_threshold: float, sell_threshold: float) -> int:
    """
    Three-way target:
     1 = Buy
     0 = Hold
    -1 = Sell
    """
    if pd.isna(forward_return):
        return np.nan
    if forward_return >= buy_threshold:
        return 1
    if forward_return <= sell_threshold:
        return -1
    return 0


def classify_binary_high_conviction(forward_return: float, threshold: float) -> int:
    """
    Binary target for stronger upward moves only.
    """
    if pd.isna(forward_return):
        return np.nan
    return int(forward_return >= threshold)


def build_regime_shift_flag(current_regime: pd.Series) -> pd.Series:
    """
    Flag whether the regime proxy changes on the next day.
    """
    return (current_regime != current_regime.shift(-1)).astype(float)


def safe_rank_pct(series: pd.Series) -> pd.Series:
    return series.rank(pct=True)

In [5]:
# ============================================
# Cell 5: Build optimized forward-return targets
# Purpose:
# - Construct targets that are less noisy than the original next-day formulation
# - Focus on 5-day and 10-day horizons that better match market structure and execution
# - Keep all targets strictly forward-looking
# ============================================

target_df = df.copy()

# -----------------------------
# Core forward returns
# -----------------------------
target_df["target_return_1d"] = target_df["spy_close"].pct_change().shift(-1)
target_df["target_return_5d"] = target_df["spy_close"].pct_change(5).shift(-5)
target_df["target_return_10d"] = target_df["spy_close"].pct_change(10).shift(-10)

# -----------------------------
# Excess-return style targets relative to cash proxy
# Here cash is approximated as zero daily excess because short-horizon daily cash drag is negligible.
# This keeps the target aligned with the trading objective.
# -----------------------------
target_df["target_excess_return_5d"] = target_df["target_return_5d"]
target_df["target_excess_return_10d"] = target_df["target_return_10d"]

print("Optimized forward-return targets created.")
print(
    target_df[
        [
            "target_return_1d",
            "target_return_5d",
            "target_return_10d",
            "target_excess_return_5d",
            "target_excess_return_10d",
        ]
    ].head()
)

Optimized forward-return targets created.
            target_return_1d  target_return_5d  target_return_10d  \
Date                                                                
2015-02-02          0.014461          0.013421           0.040561   
2015-02-03         -0.003808          0.009617           0.025825   
2015-02-04          0.010095          0.014064           0.029011   
2015-02-05         -0.002765          0.013584           0.024840   
2015-02-06         -0.004476          0.020579           0.027536   

            target_excess_return_5d  target_excess_return_10d  
Date                                                           
2015-02-02                 0.013421                  0.040561  
2015-02-03                 0.009617                  0.025825  
2015-02-04                 0.014064                  0.029011  
2015-02-05                 0.013584                  0.024840  
2015-02-06                 0.020579                  0.027536  


In [6]:
# ============================================
# Cell 6: Build higher-conviction classification targets
# Purpose:
# - Reduce label noise by focusing on moves large enough to matter
# - Create better targets for machine learning classification
# - Preserve a clean mapping between direction, strength, and actionability
# ============================================

# Thresholds chosen to create more informative labels than naive next-day direction.
BUY_THRESHOLD_5D = 0.005
SELL_THRESHOLD_5D = -0.005

BUY_THRESHOLD_10D = 0.010
SELL_THRESHOLD_10D = -0.010

target_df["target_direction_1d"] = (target_df["target_return_1d"] > 0).astype(float)
target_df["target_direction_5d"] = (target_df["target_return_5d"] > 0).astype(float)
target_df["target_direction_10d"] = (target_df["target_return_10d"] > 0).astype(float)

target_df["target_high_conviction_up_5d"] = target_df["target_return_5d"].apply(
    lambda x: classify_binary_high_conviction(x, BUY_THRESHOLD_5D)
)
target_df["target_high_conviction_up_10d"] = target_df["target_return_10d"].apply(
    lambda x: classify_binary_high_conviction(x, BUY_THRESHOLD_10D)
)

target_df["target_signal_5d"] = target_df["target_return_5d"].apply(
    lambda x: classify_three_way_signal(x, BUY_THRESHOLD_5D, SELL_THRESHOLD_5D)
)
target_df["target_signal_10d"] = target_df["target_return_10d"].apply(
    lambda x: classify_three_way_signal(x, BUY_THRESHOLD_10D, SELL_THRESHOLD_10D)
)

print("Higher-conviction classification targets created.")
print(
    target_df[
        [
            "target_direction_1d",
            "target_direction_5d",
            "target_direction_10d",
            "target_high_conviction_up_5d",
            "target_high_conviction_up_10d",
            "target_signal_5d",
            "target_signal_10d",
        ]
    ].head()
)

Higher-conviction classification targets created.
            target_direction_1d  target_direction_5d  target_direction_10d  \
Date                                                                         
2015-02-02                  1.0                  1.0                   1.0   
2015-02-03                  0.0                  1.0                   1.0   
2015-02-04                  1.0                  1.0                   1.0   
2015-02-05                  0.0                  1.0                   1.0   
2015-02-06                  0.0                  1.0                   1.0   

            target_high_conviction_up_5d  target_high_conviction_up_10d  \
Date                                                                      
2015-02-02                           1.0                            1.0   
2015-02-03                           1.0                            1.0   
2015-02-04                           1.0                            1.0   
2015-02-05                  

In [7]:
# ============================================
# Cell 7: Build regime-aware and trend-conditional targets
# Purpose:
# - Give the model cleaner labels that respect market context
# - Reduce the confusion created by mixing all market states together
# - Provide additional learning targets for later experimentation
# ============================================

target_df["target_above_200d_now"] = (target_df["spy_close"] > target_df["price_to_sma_200"].add(1.0).mul(
    target_df["spy_close"] / (target_df["price_to_sma_200"] + 1.0)
)).astype(float) if "price_to_sma_200" in target_df.columns else np.nan

# More robust trend-conditional proxy using the selected optimized feature itself.
# If price_to_sma_200 > 0, market is above the 200-day moving average.
if "price_to_sma_200" in target_df.columns:
    target_df["target_above_200d_now"] = (target_df["price_to_sma_200"] > 0).astype(float)
else:
    target_df["target_above_200d_now"] = np.nan

target_df["target_trend_conditional_up_5d"] = (
    (target_df["target_return_5d"] >= BUY_THRESHOLD_5D) &
    (target_df["target_above_200d_now"] == 1)
).astype(float)

target_df["target_trend_conditional_up_10d"] = (
    (target_df["target_return_10d"] >= BUY_THRESHOLD_10D) &
    (target_df["target_above_200d_now"] == 1)
).astype(float)

target_df["target_regime_shift_flag"] = build_regime_shift_flag(target_df["phase_2_5_regime_proxy"])

print("Regime-aware and trend-conditional targets created.")
print(
    target_df[
        [
            "target_above_200d_now",
            "target_trend_conditional_up_5d",
            "target_trend_conditional_up_10d",
            "target_regime_shift_flag",
        ]
    ].head()
)

Regime-aware and trend-conditional targets created.
            target_above_200d_now  target_trend_conditional_up_5d  \
Date                                                                
2015-02-02                    1.0                             1.0   
2015-02-03                    1.0                             1.0   
2015-02-04                    1.0                             1.0   
2015-02-05                    1.0                             1.0   
2015-02-06                    1.0                             1.0   

            target_trend_conditional_up_10d  target_regime_shift_flag  
Date                                                                   
2015-02-02                              1.0                       0.0  
2015-02-03                              1.0                       0.0  
2015-02-04                              1.0                       0.0  
2015-02-05                              1.0                       0.0  
2015-02-06                      

In [8]:
# ============================================
# Cell 8: Build optimized continuous trading scores
# Purpose:
# - Create smoother decision variables for later quant and ensemble layers
# - Combine future return magnitude with directional conviction
# - Provide richer labels than discrete classes alone
# ============================================

# Trading score is intentionally continuous and forward-looking.
# It can later be used for ranking opportunities or calibrating exposure.
target_df["target_score_5d"] = target_df["target_return_5d"]
target_df["target_score_10d"] = target_df["target_return_10d"]

# Conviction-weighted score:
# positive strong returns score higher; weak/noisy cases remain near zero.
target_df["target_conviction_score_5d"] = (
    target_df["target_return_5d"] *
    (target_df["target_high_conviction_up_5d"].fillna(0.0) + 1.0)
)

target_df["target_conviction_score_10d"] = (
    target_df["target_return_10d"] *
    (target_df["target_high_conviction_up_10d"].fillna(0.0) + 1.0)
)

print("Continuous trading scores created.")
print(
    target_df[
        [
            "target_score_5d",
            "target_score_10d",
            "target_conviction_score_5d",
            "target_conviction_score_10d",
        ]
    ].head()
)

Continuous trading scores created.
            target_score_5d  target_score_10d  target_conviction_score_5d  \
Date                                                                        
2015-02-02         0.013421          0.040561                    0.026842   
2015-02-03         0.009617          0.025825                    0.019235   
2015-02-04         0.014064          0.029011                    0.028129   
2015-02-05         0.013584          0.024840                    0.027169   
2015-02-06         0.020579          0.027536                    0.041158   

            target_conviction_score_10d  
Date                                     
2015-02-02                     0.081121  
2015-02-03                     0.051650  
2015-02-04                     0.058022  
2015-02-05                     0.049680  
2015-02-06                     0.055072  


In [9]:
# ============================================
# Cell 9: Remove forward-looking leakage rows and finalize the optimized target dataset
# Purpose:
# - Remove rows created by forward shifts
# - Keep the dataset aligned and ready for the next optimized modeling cycle
# - Preserve only valid rows inside the defined research horizon
# ============================================

required_target_cols = [
    "target_return_1d",
    "target_return_5d",
    "target_return_10d",
    "target_high_conviction_up_5d",
    "target_high_conviction_up_10d",
    "target_signal_5d",
    "target_signal_10d",
    "target_trend_conditional_up_5d",
    "target_trend_conditional_up_10d",
    "target_score_5d",
    "target_score_10d",
]

target_df = target_df.dropna(subset=required_target_cols).copy()

target_df = target_df[
    (target_df.index >= pd.Timestamp(TRAIN_START)) &
    (target_df.index <= pd.Timestamp(LIVE_END))
].copy()

target_df["is_train_period"] = (
    (target_df.index >= pd.Timestamp(TRAIN_START)) &
    (target_df.index <= pd.Timestamp(TRAIN_END))
)

target_df["is_live_period"] = (
    (target_df.index >= pd.Timestamp(LIVE_START)) &
    (target_df.index <= pd.Timestamp(LIVE_END))
)

target_df["dataset_partition"] = np.select(
    [
        target_df["is_train_period"],
        target_df["is_live_period"],
    ],
    [
        "train",
        "live",
    ],
    default="unused"
)

print("Optimized target dataset finalized.")
print("Final dataset shape:", target_df.shape)
print("\nPartition counts:")
print(target_df["dataset_partition"].value_counts(dropna=False))

Optimized target dataset finalized.
Final dataset shape: (2770, 73)

Partition counts:
dataset_partition
train    2496
live      274
Name: count, dtype: int64


In [10]:
# ============================================
# Cell 10: Validate the optimized target layer
# Purpose:
# - Confirm the new target set is structurally sound
# - Check for missing values and duplicates
# - Inspect class balance before moving into the optimized quant and ML pipeline
# ============================================

assert target_df.index.is_monotonic_increasing, "Target dataset index must be sorted ascending."
assert target_df.index.duplicated().sum() == 0, "Target dataset contains duplicate timestamps."

validation_target_cols = [
    "target_return_1d",
    "target_return_5d",
    "target_return_10d",
    "target_direction_1d",
    "target_direction_5d",
    "target_direction_10d",
    "target_high_conviction_up_5d",
    "target_high_conviction_up_10d",
    "target_signal_5d",
    "target_signal_10d",
    "target_score_5d",
    "target_score_10d",
]

missing_target_values = target_df[validation_target_cols].isna().sum().sum()
if missing_target_values != 0:
    raise ValueError(f"Optimized targets still contain missing values: {missing_target_values}")

print("Optimized target validation passed.")

print("\nTarget distribution: target_high_conviction_up_5d")
print(target_df["target_high_conviction_up_5d"].value_counts(dropna=False))

print("\nTarget distribution: target_high_conviction_up_10d")
print(target_df["target_high_conviction_up_10d"].value_counts(dropna=False))

print("\nSignal distribution: target_signal_5d")
print(target_df["target_signal_5d"].value_counts(dropna=False))

print("\nSignal distribution: target_signal_10d")
print(target_df["target_signal_10d"].value_counts(dropna=False))

Optimized target validation passed.

Target distribution: target_high_conviction_up_5d
target_high_conviction_up_5d
0.0    1442
1.0    1328
Name: count, dtype: int64

Target distribution: target_high_conviction_up_10d
target_high_conviction_up_10d
0.0    1512
1.0    1258
Name: count, dtype: int64

Signal distribution: target_signal_5d
target_signal_5d
 1.0    1328
-1.0     790
 0.0     652
Name: count, dtype: int64

Signal distribution: target_signal_10d
target_signal_10d
 1.0    1258
 0.0     868
-1.0     644
Name: count, dtype: int64


In [11]:
# ============================================
# Cell 11: Save Phase 3.1 outputs for the optimized Phase 4 / 5 cycle
# Purpose:
# - Persist the optimized target dataset
# - Save metadata for reproducibility
# - Create a stable contract for the next optimized notebooks
# ============================================

phase_3_1_csv = PROCESSED_DIR / "phase_3_1_target_data.csv"
phase_3_1_parquet = PROCESSED_DIR / "phase_3_1_target_data.parquet"

target_df.to_csv(phase_3_1_csv, index=True)
print("Saved CSV:", phase_3_1_csv)

try:
    target_df.to_parquet(phase_3_1_parquet, index=True)
    print("Saved Parquet:", phase_3_1_parquet)
except Exception as e:
    print("Parquet save skipped:", e)

metadata = {
    "phase": "Phase 3.1 - Target Construction Layer (Optimized)",
    "source_notebook": "3.1 - Target Construction Layer (Optimized).ipynb",
    "rows": int(len(target_df)),
    "columns": target_df.columns.tolist(),
    "start_date": str(target_df.index.min().date()),
    "end_date": str(target_df.index.max().date()),
    "buy_threshold_5d": float(BUY_THRESHOLD_5D),
    "sell_threshold_5d": float(SELL_THRESHOLD_5D),
    "buy_threshold_10d": float(BUY_THRESHOLD_10D),
    "sell_threshold_10d": float(SELL_THRESHOLD_10D),
}

phase_3_1_metadata_path = META_DIR / "phase_3_1_metadata.json"
with open(phase_3_1_metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("Saved metadata:", phase_3_1_metadata_path)
print("Phase 3.1 complete.")

Saved CSV: c:\Users\Tyler\Regime-Aware Risk Integrated Alpha\data\processed\phase_3_1_target_data.csv
Saved Parquet: c:\Users\Tyler\Regime-Aware Risk Integrated Alpha\data\processed\phase_3_1_target_data.parquet
Saved metadata: c:\Users\Tyler\Regime-Aware Risk Integrated Alpha\reports\metadata\phase_3_1_metadata.json
Phase 3.1 complete.


In [12]:
# ============================================
# Cell 12: Phase 4.1 / 5.1 loader snippet
# Purpose:
# - Provide a stable loading pattern for the optimized downstream notebooks
# - Keep notebook integration deterministic
# - Confirm the optimized target handoff works
# ============================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

phase_3_1_parquet = PROCESSED_DIR / "phase_3_1_target_data.parquet"
phase_3_1_csv = PROCESSED_DIR / "phase_3_1_target_data.csv"

print("Looking for Phase 3.1 files:")
print("Parquet:", phase_3_1_parquet)
print("CSV:", phase_3_1_csv)

if phase_3_1_parquet.exists():
    phase_3_1_data = pd.read_parquet(phase_3_1_parquet)
    print("Loaded Parquet file")
elif phase_3_1_csv.exists():
    phase_3_1_data = pd.read_csv(phase_3_1_csv, index_col=0, parse_dates=True)
    print("Loaded CSV file")
else:
    raise FileNotFoundError(
        "Phase 3.1 data file was not found. "
        f"Expected one of:\n{phase_3_1_parquet}\n{phase_3_1_csv}"
    )

phase_3_1_data.index = pd.to_datetime(phase_3_1_data.index)
phase_3_1_data = phase_3_1_data.sort_index()

print("\nLoaded Phase 3.1 dataset preview:")
print(phase_3_1_data.head())

Looking for Phase 3.1 files:
Parquet: c:\Users\Tyler\Regime-Aware Risk Integrated Alpha\data\processed\phase_3_1_target_data.parquet
CSV: c:\Users\Tyler\Regime-Aware Risk Integrated Alpha\data\processed\phase_3_1_target_data.csv
Loaded Parquet file

Loaded Phase 3.1 dataset preview:
              spy_open    spy_high     spy_low   spy_close  spy_adj_close  \
Date                                                                        
2015-02-02  200.050003  202.029999  197.860001  201.919998     167.218262   
2015-02-03  203.000000  204.850006  202.550003  204.839996     169.636398   
2015-02-04  203.919998  205.380005  203.509995  204.059998     168.990479   
2015-02-05  204.860001  206.300003  204.770004  206.119995     170.696411   
2015-02-06  206.559998  207.240005  204.919998  205.550003     170.224442   

            spy_volume  vix_close  tnx_close  is_train_period  is_live_period  \
Date                                                                            
2015-02-02   1